## **RecycleVision- Garbage Image Classification**

# 1) Libraries import + path set

In [28]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import os

datadir = "/content/drive/MyDrive/RecycleVision- Garbage Image Classification/Data"

img_size = (224, 224)
batch_size = 32
num_classes = 6

# 2) Dataset load (train / val split)

In [29]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    datadir,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=img_size,
    batch_size=batch_size,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    datadir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=img_size,
    batch_size=batch_size,
)

class_names = train_ds.class_names
print("Classes:", class_names)

Found 2537 files belonging to 6 classes.
Using 2030 files for training.
Found 2537 files belonging to 6 classes.
Using 507 files for validation.
Classes: ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']


In [30]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

# 3) Preprocessing + data augmentation*

In [31]:
data_augmentation = keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.1),
        layers.RandomZoom(0.1),
    ]
)
preprocess_input = keras.applications.mobilenet_v2.preprocess_input

# 4) MobileNetV2 base model

In [32]:
IMG_SIZE = img_size
INPUT_SHAPE = IMG_SIZE + (3,)

base_model = keras.applications.MobileNetV2(
    input_shape=INPUT_SHAPE,
    include_top=False,
    weights="imagenet",
)
base_model.trainable = False

global_average_layer = layers.GlobalAveragePooling2D()
prediction_layer = layers.Dense(num_classes, activation="softmax")

# 5) Final model

In [33]:
inputs = keras.Input(shape=INPUT_SHAPE)
x = data_augmentation(inputs)
x = preprocess_input(x)
x = base_model(x, training=False)
x = global_average_layer(x)
x = layers.Dropout(0.2)(x)
outputs = prediction_layer(x)

model = keras.Model(inputs, outputs)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_1 (Sequential)       │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │         7,686 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,265,670 (8.64 MB)

 Trainable params: 7,686 (30.02 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [38]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[early_stop],
)

Epoch 1/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 132s 2s/step - accuracy: 0.8709 - loss: 0.3484 - val_accuracy: 0.8462 - val_loss: 0.4535
Epoch 2/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 149s 2s/step - accuracy: 0.8648 - loss: 0.3771 - val_accuracy: 0.8521 - val_loss: 0.4501
Epoch 3/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 149s 2s/step - accuracy: 0.8695 - loss: 0.3637 - val_accuracy: 0.8501 - val_loss: 0.4525
Epoch 4/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 132s 2s/step - accuracy: 0.8482 - loss: 0.4027 - val_accuracy: 0.8383 - val_loss: 0.4570
Epoch 5/10
64/64 ━━━━━━━━━━━━━━━━━━━━ 129s 2s/step - accuracy: 0.8553 - loss: 0.3651 - val_accuracy: 0.8481 - val_loss: 0.4562


In [40]:
loss, acc = model.evaluate(val_ds)
print("Validation accuracy:", acc)

16/16 ━━━━━━━━━━━━━━━━━━━━ 23s 1s/step - accuracy: 0.8487 - loss: 0.4593
Validation accuracy: 0.8520709872245789


In [41]:
import os

save_dir = "/content/drive/MyDrive/RecycleVision- Garbage Image Classification/Models"
os.makedirs(save_dir, exist_ok=True)

model_path = os.path.join(save_dir, "recyclevision_mobilenetv2.keras")
model.save(model_path)   # .keras format
print("Saved:", model_path)

Saved: /content/drive/MyDrive/RecycleVision- Garbage Image Classification/Models/recyclevision_mobilenetv2.keras
